# Inflorescences

In PalmSim inflorescences are modelled as virtual inflorescences. This is to say that during each simulation run virtual inflorescences initiate, grow, abort, etc. 

### Simulating a single inflorescence

This example takes you a bit under the hood of PalmSim. Namely, we will simulate a single inflorescence.

Since this is a bit un-usual you will be exposed to some Python code which should in fact be still very read-able.

What you will see in this example is the development of an inflorescence from initiation to bunch!

In [ ]:
# Make sure we can load PalmSim and it's component from this notebook
import sys
sys.path.append('../PalmSim')
sys.path.append('../PalmSim/components')

In [ ]:
from generative import Indeterminate
infl = Indeterminate()
infl._assim_growth = 5

We now set-up and run a simulation explicitly:

In [ ]:
# import standard python libraries
import numpy as np
import pandas as pd

# Duration of the simulation (months)
dur = 50

# Time-step of update (month)
dt = 0.1

# Time-steps
steps = np.arange(0,50,dt)

# Results data
res = {}

# the iterative loop:
for t in steps:

    # update (grow, etc.) the inflorescence a dt-time-step
    infl.update(dt)
    
    # differentiate to a female if necessary
    if (infl.sex =='indeterminate') and (infl.age > infl.age_of_differentiation):
        infl = infl.to_female()
    
    # ouput data on the inflorescence
    res[t] = infl.to_comprehensive_dict()
    
    # stop the simulation if past maturity
    if (infl.sex == 'female') and (infl.age > infl._age_of_maturity + 1):
        break
        
res = pd.DataFrame(res).T
res.columns = [x.replace('_',' ') for x in list(res)]

## Let's view the simulation result

### Bunch component mass

First of all, let's consider the mass of the components:

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-colorblind')

f,ax = plt.subplots(figsize=(12,4))

res[['stalk mass (kg DM)',
     'kernel mass (kg DM)',
     'mesocarp fibers mass (kg DM)',
     'mesocarp oil mass (kg DM)']].plot.area(ax=ax)

ax.set_ylabel('Mass (kg DM)')
ax.set_xlabel('Age (months after initiation)')
ax.set_xlim(0,38);

#plt.savefig('../Documentation/images/bunch_component_mass.png',dpi=300)

In [ ]:
print('{:<20} | {:>12} | {:>12}'.format('Name','Mass (kg DM)','Mass Fraction (DM)'))
print('-------------------------------------------------------------')
for component in infl.components:
    print('{:<20} | {:>12.3f} | {:>12.3f}'.format(component._name,component.mass,component.mass/infl.mass))
    

### Abortion
Notice the two abortion sensitive periods

In [ ]:
res['abortion fraction (1)'].plot()

### Assimilation

A peak of assimilation during bunch filling!

In [ ]:
res['assim growth organ (kg CH2O/organ/month)'].plot()
res['sink strength potential (kg CH2O/organ/month)'].plot()

### Bunch potential

Consider the potential fraction bunches of per cohort over time (also described in Woittiez et al., 2017).

The fraction of inflorescences that become a bunch depends on the female:male sex ratio (month 11), the inflorescence abortion before anthesis (month 30) and the bunch failure after anthesis (month 35).

This is nicely depicted in the following figure.

In [ ]:
planting_density = 140 # per hectare
inflorescence_initiation_rate = 2 # per palm per month
potential_bunch_count_per_month = planting_density * inflorescence_initiation_rate
(potential_bunch_count_per_month*res['multiplicity (1/cohort)']).plot()
ax = plt.gca()
ax.set_xlabel('Months after initiation (months)')
ax.set_ylabel('Potential bunch count')

### Potential Bunch Mass

We have modelled the potential bunch mass to be a function of palm age.

For the stalk alone for example:

In [ ]:
import matplotlib as mpl
mpl.style.use('seaborn-colorblind')

f,ax = plt.subplots(figsize=(8,4))

y = res['stalk mass (kg DM)']
y2 = 10*res['stalk mass (kg DM)']
ax.plot(y,label='3 years after planting',color='black')
ax.plot(y2,label='30 years after planting',color='black',linestyle='dashed')

ax.legend(loc=2)
ax.set_ylabel('Potential Mass (kg DM)')
ax.set_xlabel('Age (months after initiation)')
ax.set_xlim(0,33)
ax.set_ylim(0,6)
ax.set_xticklabels(np.array([int(x) for x in ax.get_xticks()]))
ax.set_title('Stalk Mass')
None
#plt.savefig('../Documentation/images/stalk_mass.png',dpi=300)

# Bunch count

The bunch count is primarily determined by the frond initiation rate (and thus inflorescence initiation rate) and the sex ratio.

These both go down with age: less inflorescences overall, and less females w.r.t males. 

We take these trends into account as is visible in the following graphs:
    - frond initiation rate as a function of plantation age (time)
    - sex ratio as a function of plantation age (time)
    

In [ ]:
from PalmField import PalmField

p = PalmField()
df = p.run()

In [ ]:
df['organs frond initiation rate (1/palm/month)'].plot(title='Frond initiation over time (1/palm/month)')

In [ ]:
df['organs female fraction (1)'].plot(title='Sex ratio over time (female:male)')

# Bunch weight

As the number of bunches goes down over time, bunch weight increases:

In [ ]:
df['organs bunch weight (kg DM)'].plot(title='Mean bunch weight (kg DM)')

We have a peak in bunch weight for the first bunches because we over-estimate photo-synthesis around this period --- oil palm regulates photosynthesis to meet demand. This a scientifically interesting feature, however a detail. We can make this more accurate as we go along.

Note, this increase emerges from the assumed decrease in bunch count and the amount of assimilates for bunch growth!

In [ ]:
df['organs bunch count (1/ha/month)'].plot(title='Bunch count (1/ha/month)')

In [ ]:
df['organs assim growth (t/ha/month)'].plot(title='Assimilates for generative growth (t/ha/month)')

# Conclusion

The generative growth is modelled in a way that is accurate and useful for calculating yield on a monthly time resolution. This concludes this example.